## 利用 GPT-4o 视觉模态优化检索增强生成

在处理包含大量图像、图形和表格的文档时，实现检索增强生成（Retrieval-Augmented Generation，RAG）会面临独特的挑战。传统 RAG 模型擅长处理文本数据，但当视觉元素在传递信息中起关键作用时，它们往往力不从心。在本 cookbook 中，我们利用视觉模态来提取和解读视觉内容，从而弥合这一差距，确保生成的回答尽可能信息丰富且准确。

我们的方法是将文档解析为图像，并利用元数据标记来识别包含图像、图形和表格的页面。当语义搜索检索到此类页面时，我们会将页面图像传给视觉模型，而不是仅依赖文本。这种方法增强了模型理解和回答与视觉数据相关的用户查询的能力。

在本 cookbook 中，我们将探索并演示以下关键概念：

##### 1. 使用 Pinecone 设置向量存储：
- 了解如何初始化和配置 Pinecone，以高效存储向量嵌入。

##### 2. 解析 PDF 并提取视觉信息：
- 探索将 PDF 页面转换为图像的技术。
- 使用 GPT-4o 视觉模态从包含图像、图形或表格的页面中提取文本信息。  

##### 3. 生成嵌入： 
- 利用嵌入模型为文本数据创建向量表示。 
- 标记包含视觉内容的页面，以便在向量存储中设置元数据标志，并检索图像，通过视觉模态传给 GPT-4o。 

##### 4. 将嵌入上传到 Pinecone： 
- 将这些嵌入上传到 Pinecone 以进行存储和检索。 

##### 5. 执行语义搜索以查找相关页面：
- 对页面文本实现语义搜索，找出与用户查询最匹配的页面。 
- 将匹配页面的文本作为上下文提供给 GPT-4o，以回答用户的查询。  

##### 6. 处理包含视觉内容的页面（可选步骤）：
- 了解如何通过 GPT-4o 视觉模态传入图像，以借助额外上下文进行问答。 
- 理解这一过程如何提升涉及视觉数据的回答的准确性。
 
完成本 cookbook 后，你将深入理解如何实现能够处理和解读包含复杂视觉元素的文档的 RAG 系统。这些知识将帮助你构建能提供更丰富、更准确信息的 AI 解决方案，从而提升用户满意度和参与度。

我们将使用世界银行的报告——[A Better Bank for a Better World: Annual Report 2024](https://documents1.worldbank.org/curated/en/099101824180532047/pdf/BOSIB13bdde89d07f1b3711dd8e86adb477.pdf) 来说明这些概念，因为该文档混合了图像、表格和图形数据。 

请记住，使用视觉模态会消耗大量资源，导致延迟和成本增加。建议仅在纯文本提取方法在评估基准上的表现不尽如人意时才使用视觉模态。了解这些背景后，让我们开始吧。

### 第 1 步：使用 Pinecone 设置向量存储 
在本节中，我们将使用 Pinecone 设置一个向量存储，以高效地存储和管理嵌入。Pinecone 是一个针对高维向量数据优化的向量数据库，这对于语义搜索和相似度匹配等任务至关重要。

**前置条件** 
1. 按照 [Pinecone Database Quickstart](https://docs.pinecone.io/guides/get-started/quickstart) 中的说明注册 Pinecone 并获取 API 密钥  
2. 使用 `pip install "pinecone[grpc]"` 安装 Pinecone SDK。gRPC（gRPC Remote Procedure Call）是一个高性能、开源的通用 RPC 框架，它使用 HTTP/2 进行传输，使用 Protocol Buffers（protobuf）作为接口定义语言，并支持分布式系统中的客户端-服务器通信。它旨在让服务间通信更加高效，适用于微服务架构。   

**安全地存储 API 密钥**  
1. 出于安全考虑，请按如下方式将 API 密钥存储在项目目录下的 .env 文件中：  
 `PINECONE_API_KEY=your-api-key-here`。   
 2. 安装 `pip install python-dotenv`，以便从 .env 文件中读取 API 密钥。 

**创建 Pinecone 索引**   
我们将使用 `create_index` 函数在 Pinecone 上初始化嵌入数据库。需要考虑两个关键参数：

1. 维度（Dimension）：必须与所选模型生成的嵌入维度一致。例如，OpenAI 的 text-embedding-ada-002 模型生成 1536 维的嵌入，而 text-embedding-3-large 生成 3072 维的嵌入。我们将使用 text-embedding-3-large 模型，因此将维度设置为 3072。

2. 度量（Metric）：距离度量决定了如何计算向量之间的相似度。Pinecone 支持多种度量，包括 cosine、dotproduct 和 euclidean。在本 cookbook 中，我们将使用余弦相似度度量。你可以在 [Pinecone 距离度量文档](https://docs.pinecone.io/guides/indexes/understanding-indexes#distance-metrics)中了解更多关于距离度量的信息。

In [9]:
import os
import time
# Import the Pinecone library
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

# Initialize a Pinecone client with your API key
pc = Pinecone(api_key)

# Create a serverless index
index_name = "my-test-index"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=3072,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

前往 [Pinecone](https://app.pinecone.io/) 上的 Indexes 列表，你应该能在索引列表中看到 `my-test-index`。 

### 第 2 步：解析 PDF 并提取视觉信息：

在本节中，我们将解析 PDF 文档——世界银行报告 [A Better Bank for a Better World: Annual Report 2024](https://documents1.worldbank.org/curated/en/099101824180532047/pdf/BOSIB13bdde89d07f1b3711dd8e86adb477.pdf)，并提取文本和视觉信息，例如对图像、图形和表格的描述。该过程包括三个主要步骤：

1. **将 PDF 解析为单独的页面：** 我们将 PDF 拆分为单独的页面，以便于处理。
2. **将 PDF 页面转换为图像：** 这使 GPT-4o 的视觉能力可以将页面作为图像进行分析。  
3. **处理图像和表格：** 向 GPT-4o 提供指令，让其提取文本，并描述文档中的图像、图形或表格。 

**前置条件**

在继续之前，请确保已安装以下软件包，并确保已将 OpenAI API 密钥设置为环境变量。你可能还需要安装 Poppler 来渲染 PDF。  

`pip install PyPDF2 pdf2image pytesseract pandas tqdm`
 
**步骤分解：**

**1. 下载 PDF 并分块：**  
- `chunk_document` 函数从提供的 URL 下载 PDF，并使用 PyPDF2 将其拆分为单独的页面。
- 每个页面作为单独的 PDF 字节流存储在一个列表中。

**2. 将 PDF 页面转换为图像：** 
- `convert_page_to_image` 函数接收单个页面的 PDF 字节，并使用 pdf2image 将其转换为图像。
- 图像会保存在本地的 'images' 目录中，以供后续处理。

**3. 使用 GPT-4o 视觉模态提取文本：**
- `extract_text_from_image` 函数使用 GPT-4o 的视觉能力从页面图像中提取文本。
- 这种方法甚至可以从扫描文档中提取文本信息。
- 请注意，这种模态资源消耗较大，因此延迟和成本也更高。 

**4. 处理整个文档：** 
- process_document 函数负责编排每个页面的处理。
- 它使用进度条（tqdm）显示处理状态。
- 从每个页面提取的信息会被收集到一个列表中，然后转换为 Pandas DataFrame。

In [7]:
import base64
import requests
import os
import pandas as pd
from PyPDF2 import PdfReader, PdfWriter
from pdf2image import convert_from_bytes
from io import BytesIO
from openai import OpenAI
from tqdm import tqdm

# Link to the document we will use as the example 
document_to_parse = "https://documents1.worldbank.org/curated/en/099101824180532047/pdf/BOSIB13bdde89d07f1b3711dd8e86adb477.pdf"

# OpenAI client 
oai_client = OpenAI()


# Chunk the PDF document into single page chunks 
def chunk_document(document_url):
    # Download the PDF document
    response = requests.get(document_url)
    pdf_data = response.content

    # Read the PDF data using PyPDF2
    pdf_reader = PdfReader(BytesIO(pdf_data))
    page_chunks = []

    for page_number, page in enumerate(pdf_reader.pages, start=1):
        pdf_writer = PdfWriter()
        pdf_writer.add_page(page)
        pdf_bytes_io = BytesIO()
        pdf_writer.write(pdf_bytes_io)
        pdf_bytes_io.seek(0)
        pdf_bytes = pdf_bytes_io.read()
        page_chunk = {
            'pageNumber': page_number,
            'pdfBytes': pdf_bytes
        }
        page_chunks.append(page_chunk)

    return page_chunks


# Function to encode the image
def encode_image(local_image_path):
    with open(local_image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


# Function to convert page to image     
def convert_page_to_image(pdf_bytes, page_number):
    # Convert the PDF page to an image
    images = convert_from_bytes(pdf_bytes)
    image = images[0]  # There should be only one page

    # Define the directory to save images (relative to your script)
    images_dir = 'images'  # Use relative path here

    # Ensure the directory exists
    os.makedirs(images_dir, exist_ok=True)

    # Save the image to the images directory
    image_file_name = f"page_{page_number}.png"
    image_file_path = os.path.join(images_dir, image_file_name)
    image.save(image_file_path, 'PNG')

    # Return the relative image path
    return image_file_path


# Pass the image to the LLM for interpretation  
def get_vision_response(prompt, image_path):
    # Getting the base64 string
    base64_image = encode_image(image_path)

    response = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        },
                    },
                ],
            }
        ],
    )
    return response


# Process document function that brings it all together 
def process_document(document_url):
    try:
        # Update document status to 'Processing'
        print("Document processing started")

        # Get per-page chunks
        page_chunks = chunk_document(document_url)
        total_pages = len(page_chunks)

        # Prepare a list to collect page data
        page_data_list = []

        # Add progress bar here
        for page_chunk in tqdm(page_chunks, total=total_pages, desc='Processing Pages'):
            page_number = page_chunk['pageNumber']
            pdf_bytes = page_chunk['pdfBytes']

            # Convert page to image
            image_path = convert_page_to_image(pdf_bytes, page_number)

            # Prepare question for vision API
            system_prompt = (
                "The user will provide you an image of a document file. Perform the following actions: "
                "1. Transcribe the text on the page. **TRANSCRIPTION OF THE TEXT:**"
                "2. If there is a chart, describe the image and include the text **DESCRIPTION OF THE IMAGE OR CHART**"
                "3. If there is a table, transcribe the table and include the text **TRANSCRIPTION OF THE TABLE**"
            )

            # Get vision API response
            vision_response = get_vision_response(system_prompt, image_path)

            # Extract text from vision response
            text = vision_response.choices[0].message.content

            # Collect page data
            page_data = {
                'PageNumber': page_number,
                'ImagePath': image_path,
                'PageText': text
            }
            page_data_list.append(page_data)

        # Create DataFrame from page data
        pdf_df = pd.DataFrame(page_data_list)
        print("Document processing completed.")
        print("DataFrame created with page data.")

        # Return the DataFrame
        return pdf_df

    except Exception as err:
        print(f"Error processing document: {err}")
        # Update document status to 'Error'


df = process_document(document_to_parse)

Document processing started


Processing Pages: 100%|██████████| 49/49 [18:54<00:00, 23.14s/it]

Document processing completed.
DataFrame created with page data.


让我们检查一下 DataFrame，确保页面已被正确处理。为简洁起见，我们只获取并显示前五行。此外，你应该能在 'images' 目录中看到生成的页面图像。

In [10]:
from IPython.display import display, HTML

# Convert the DataFrame to an HTML table and display top 5 rows 
display(HTML(df.head().to_html()))

让我们看一个示例页面，比如第 21 页，它包含嵌入的图形和文本。可以看到，视觉模态有效地提取并描述了视觉信息。例如，该页面上的饼图被准确地描述为：

`"FIGURE 6: MIDDLE EAST AND NORTH AFRICA IBRD AND IDA LENDING BY SECTOR - FISCAL 2024 SHARE OF TOTAL OF $4.6 BILLION" is a circular chart, resembling a pie chart, illustrating the percentage distribution of funds among different sectors. The sectors include:`

In [11]:
# Filter and print rows where pageNumber is 21
filtered_rows = df[df['PageNumber'] == 21]
for text in filtered_rows.PageText:
    print(text)

**TRANSCRIPTION OF THE TEXT:**

We also committed $35 million in grants to support emergency relief in Gaza. Working with the World Food Programme, the World Health Organization, and the UN Children’s Fund, the grants supported the delivery of emergency food, water, and medical supplies. In the West Bank, we approved a $200 million program for the continuation of education for children, $22 million to support municipal services, and $45 million to strengthen healthcare and hospital services.

**Enabling green and resilient growth**
To help policymakers in the region advance their climate change and development goals, we published Country Climate and Development Reports for the West Bank and Gaza, Lebanon, and Tunisia. In Libya, the catastrophic flooding in September 2023 devastated eastern localities, particularly the city of Derna. The World Bank, together with the UN and the European Union, produced a Rapid Damage and Needs Assessment to inform recovery and reconstruction efforts.

W

### 第 3 步：生成嵌入： 

在本节中，我们专注于将从文档每一页提取的文本内容转换为向量嵌入。这些嵌入捕捉了文本的语义，从而支持高效的相似度搜索和各种自然语言处理（NLP）任务。我们还会识别包含图像、图形或表格等视觉元素的页面，并对其进行标记以便特殊处理。

**步骤分解：**

**1. 为视觉内容添加标志**   
  
为了处理包含视觉信息的页面，我们在第 2 步中使用视觉模态从图表、表格和图像中提取了内容。通过在提示词中加入特定指令，我们确保模型在描述视觉内容时会添加诸如 `DESCRIPTION OF THE IMAGE OR CHART` 或 `TRANSCRIPTION OF THE TABLE` 之类的标记。在这一步中，如果检测到此类标记，我们就将 Visual_Input_Processed 标志设置为 'Y'；否则保持为 'N'。

虽然视觉模态能有效捕捉大部分视觉信息，但某些细节——尤其是在工程图纸等复杂视觉内容中——可能会在转换过程中丢失。在第 6 步中，我们将使用这个标志来决定何时将页面图像作为额外上下文传给 GPT-4 Vision。这是一项可选的增强措施，可以显著提升 RAG 解决方案的效果。  

**2. 使用 OpenAI 的嵌入模型生成嵌入**  

我们使用 OpenAI 的嵌入模型 `text-embedding-3-large` 生成高维嵌入，以表示每个页面的语义内容。 

注意：务必确保所用嵌入模型的维度与 Pinecone 向量存储的配置一致。在我们的例子中，我们将 Pinecone 数据库设置为 3072 维，以匹配 `text-embedding-3-large` 的默认维度。  


In [112]:
# Add a column to flag pages with visual content
df['Visual_Input_Processed'] = df['PageText'].apply(
    lambda x: 'Y' if 'DESCRIPTION OF THE IMAGE OR CHART' in x or 'TRANSCRIPTION OF THE TABLE' in x else 'N'
)


# Function to get embeddings
def get_embedding(text_input):
    response = oai_client.embeddings.create(
        input=text_input,
        model="text-embedding-3-large"
    )
    return response.data[0].embedding


# Generate embeddings with a progress bar
embeddings = []
for text in tqdm(df['PageText'], desc='Generating Embeddings'):
    embedding = get_embedding(text)
    embeddings.append(embedding)

# Add the embeddings to the DataFrame
df['Embeddings'] = embeddings

Generating Embeddings: 100%|██████████| 49/49 [00:18<00:00,  2.61it/s]


我们可以验证逻辑是否正确标记了需要视觉输入的页面。例如，我们之前查看过的第 21 页，其 Visual_Input_Needed 标志被设置为 "Y"。

In [113]:
# Display the flag for page 21 
filtered_rows = df[df['PageNumber'] == 21]
print(filtered_rows.Visual_Input_Processed)

20    Y
Name: Visual_Input_Processed, dtype: object


#### 第 4 步：将嵌入上传到 Pinecone： 

在本节中，我们将把为文档每一页生成的嵌入上传到 Pinecone。除了嵌入之外，我们还会附上描述每个页面的相关元数据标签，例如页码、文本内容、图像路径以及页面是否包含图形。 

**步骤分解：**  

**1. 创建元数据字段：**  
元数据增强了我们执行更细粒度搜索、查找与向量关联的文本或图像的能力，并支持在向量数据库中进行过滤。
* pageId：将 document_id 和 pageNumber 组合起来，为每个页面创建唯一标识符。我们将其用作嵌入的唯一标识符。 
* pageNumber：页面在文档中的页码数字。
* text：从页面中提取的文本内容。
* ImagePath：与页面关联的图像文件路径。
* GraphicIncluded：一个布尔值或标志，表示页面是否包含可能需要视觉处理的图形元素。

**2. 上传嵌入：**  
我们将在 `upsert_vector` 函数中使用 Pinecone API 来 "upsert" 以下值—— 

* 唯一标识符
* 嵌入
* 上面定义的元数据 

注意："Upsert" 是 "update"（更新）和 "insert"（插入）两个词的组合。在数据库操作中，upsert 是一种原子操作：如果记录已存在则更新它，否则插入一条新记录。当你希望确保数据库拥有最新数据，又不想分别检查是插入还是更新时，这一操作尤其有用。

In [114]:
# reload the index from Pinecone 
index = pc.Index(index_name)

# Create a document ID prefix 
document_id = 'WB_Report'


# Define the async function correctly
def upsert_vector(identifier, embedding, metadata):
    try:
        index.upsert([
            {
                'id': identifier,
                'values': embedding,
                'metadata': metadata
            }
        ])
    except Exception as e:
        print(f"Error upserting vector with ID {identifier}: {e}")
        raise


for idx, row in tqdm(df.iterrows(), total=df.shape[0], desc='Uploading to Pinecone'):
    pageNumber = row['PageNumber']

    # Create meta-data tags to be added to Pinecone 
    metadata = {
        'pageId': f"{document_id}-{pageNumber}",
        'pageNumber': pageNumber,
        'text': row['PageText'],
        'ImagePath': row['ImagePath'],
        'GraphicIncluded': row['Visual_Input_Processed']
    }

    upsert_vector(metadata['pageId'], row['Embeddings'], metadata)


Uploading to Pinecone: 100%|██████████| 49/49 [00:08<00:00,  5.93it/s]


前往 [Pinecone](https://app.pinecone.io/) 上的 Indexes 列表，你应该能看到已 upsert 到数据库中的向量及其元数据。

### 第 5 步：执行语义搜索以查找相关页面：
在本节中，我们实现语义搜索，以在文档中找到能回答用户问题的最相关页面。这种方法使用存储在 Pinecone 向量数据库中的嵌入，根据页面内容与用户查询的语义相似度来检索页面。这样，我们就能有效地搜索文本内容，并将其作为上下文提供给 GPT-4o 来回答用户的问题。

**步骤分解：**  

**1. 为用户的问题生成嵌入** 

* 我们使用 OpenAI 的嵌入模型为用户的问题生成高维向量表示。 
* 该向量捕捉了问题的语义，使我们能够针对已存储的嵌入执行高效的相似度搜索。 
* 嵌入对于确保搜索查询与文档内容在语义上保持一致至关重要，即使用词并不完全相同。

**2. 查询 Pinecone 索引以查找相关页面** 

* 使用生成的嵌入，我们查询 Pinecone 索引以找到最相关的页面。 
* Pinecone 通过 `cosine` 相似度将问题的嵌入与向量数据库中存储的嵌入进行比较，从而执行相似度搜索。回想一下，我们在第 1 步创建 Pinecone 数据库时将其设置为了 `metric` 参数。 
* 我们指定要检索的最佳匹配数量，通常需要在覆盖范围和相关性之间取得平衡。例如，检索前 3-5 个页面通常就足以提供全面的答案，而不会用过多的上下文让模型不堪重负。 


**3. 汇总匹配页面的元数据以提供上下文** 

* 一旦识别出相关嵌入，我们就收集与之关联的元数据，包括提取的文本和页码。
* 这些元数据对于组织提供给 GPT-4o 的上下文至关重要。 
* 我们还将汇总的信息格式化为 JSON，以便 LLM 更容易解读。

**4. 使用 GPT-4o 模型生成答案** 

* 最后，我们将汇总的上下文传给 GPT-4o。 
* 模型利用这些上下文，为用户的问题生成信息丰富、连贯且与上下文相关的答案。
* 由于能够访问文档中的相关信息，检索到的上下文帮助 LLM 更准确地回答问题。 

In [115]:
import json


# Function to get response to a user's question 
def get_response_to_question(user_question, pc_index):
    # Get embedding of the question to find the relevant page with the information 
    question_embedding = get_embedding(user_question)

    # get response vector embeddings 
    response = pc_index.query(
        vector=question_embedding,
        top_k=2,
        include_values=True,
        include_metadata=True
    )

    # Collect the metadata from the matches
    context_metadata = [match['metadata'] for match in response['matches']]

    # Convert the list of metadata dictionaries to prompt a JSON string
    context_json = json.dumps(context_metadata, indent=3)

    prompt = f"""You are a helpful assistant. Use the following context and images to answer the question. In the answer, include the reference to the document, and page number you found the information on between <source></source> tags. If you don't find the information, you can say "I couldn't find the information"

    question: {user_question}
    
    <SOURCES>
    {context_json}
    </SOURCES>
    """

    # Call completions end point with the prompt 
    completion = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": prompt}
        ]
    )

    return completion.choices[0].message.content

现在，让我们提出一个需要从图表中获取信息的问题。在这个例子中，相关细节位于一张饼图中。

In [116]:
question = "What percentage was allocated to social protections in Western and Central Africa?"
answer = get_response_to_question(question, index)

print(answer)

Social protection was allocated 8% of the total lending in Western and Central Africa in fiscal 2024. <source>WB_Report-13, page 13</source>


让我们提高难度，提出一个需要解读表格中所呈现信息的问题。在第 2 步中，我们已使用 GPT-4o 视觉模态提取了这些信息。 

In [117]:
question = "What was the increase in access to electricity between 2000 and 2012 in Western and Central Africa?"
answer = get_response_to_question(question, index)

print(answer)

The increase in access to electricity between 2000 and 2012 in Western and Central Africa was from 34.1% to 44.1%, which is an increase of 10 percentage points. 

<source>WB_Report-13, page 13</source>


这种方法效果很好。不过，在某些情况下，信息嵌入在图像或图形中，转换为文本时会损失保真度，例如复杂的工程图纸。

通过使用 GPT-4o 视觉模态，我们可以将页面图像直接作为上下文传给模型。在下一节中，我们将探讨如何利用图像输入提升模型回答的准确性。

### 第 6 步：处理包含视觉内容的页面（可选步骤）：
当元数据表明存在图像、图形或表格时，我们可以将图像而不是提取的文本作为上下文传给 GPT-4o。当视觉信息的文本描述不足以传达上下文时，这种方法会很有用，例如工程图纸或复杂示意图等复杂图形。 

**步骤分解：** 

本步骤与第 5 步的区别在于，我们添加了额外的逻辑来识别嵌入何时设置了 `Visual_Input_Processed` 标志。在这种情况下，我们不再传入文本作为上下文，而是通过 GPT-4o 视觉模态传入页面图像作为上下文。 

注意：这种方法确实会增加延迟和成本，因为处理图像输入更耗资源、也更昂贵。因此，只有在无法通过上述第 5 步所述的纯文本模态获得理想结果时才应使用它。

In [120]:
import base64
import json


def get_response_to_question_with_images(user_question, pc_index):
    # Get embedding of the question to find the relevant page with the information 
    question_embedding = get_embedding(user_question)

    # Get response vector embeddings 
    response = pc_index.query(
        vector=question_embedding,
        top_k=3,
        include_values=True,
        include_metadata=True
    )

    # Collect the metadata from the matches
    context_metadata = [match['metadata'] for match in response['matches']]

    # Build the message content
    message_content = []

    # Add the initial prompt
    initial_prompt = f"""You are a helpful assistant. Use the text and images provided by the user to answer the question. You must include the reference to the page number or title of the section you the answer where you found the information. If you don't find the information, you can say "I couldn't find the information"

    question: {user_question}
    """
    
    message_content.append({"role": "system", "content": initial_prompt})
    
    context_messages = []

    # Process each metadata item to include text or images based on 'Visual_Input_Processed'
    for metadata in context_metadata:
        visual_flag = metadata.get('GraphicIncluded')
        page_number = metadata.get('pageNumber')
        page_text = metadata.get('text')
        message =""

        if visual_flag =='Y':
            # Include the image
            print(f"Adding page number {page_number} as an image to context")
            image_path = metadata.get('ImagePath', None)
            try:
                base64_image = encode_image(image_path)
                image_type = 'jpeg'
                # Prepare the messages for the API call
                context_messages.append({
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/{image_type};base64,{base64_image}"
                    },
                })
            except Exception as e:
                print(f"Error encoding image at {image_path}: {e}")
        else:
            # Include the text
            print(f"Adding page number {page_number} as text to context")
            context_messages.append({
                    "type": "text",
                    "text": f"Page {page_number} - {page_text}",
                })
        
                # Prepare the messages for the API call
        messages =  {
                "role": "user",
                "content": context_messages
        }
    
    message_content.append(messages)

    completion = oai_client.chat.completions.create(
    model="gpt-4o",
    messages=message_content
    )

    return completion.choices[0].message.content

让我们检查一下在第 5 步纯文本语义搜索中提出的相同问题。我们注意到，GPT-4o 模型能够识别出包含相关信息的图表来回答问题。 

In [121]:
question = "What percentage was allocated to social protections in Western and Central Africa?"
answer = get_response_to_question_with_images(question, index)

print(answer)

Adding page number 13.0 as an image to context
Adding page number 12.0 as an image to context
Adding page number 11.0 as an image to context
The percentage allocated to social protection in Western and Central Africa is 8% (Figure 2: Western and Central Africa; IBRD and IDA Lending by Sector).


现在让我们提出一个纯文本模态可能无法回答的问题，比如在文档中找到一张相关图像并对其进行描述。 

In [124]:
question = "Can you find the image associated with digital improvements and describe what you see in the images?"
answer = get_response_to_question_with_images(question, index)

print(answer)

Adding page number 32.0 as an image to context
Adding page number 10.0 as an image to context
Adding page number 4.0 as an image to context
### Image Descriptions

1. **Page 60-61 (Digital Section)**:
   - **Left Side**: A person is sitting and working on a laptop, holding a smartphone. The setting seems informal, possibly in a small office or a cafe.
   - **Text**: Discussion on scaling digital development, thought leadership, partnerships, and establishment of a Digital Vice Presidency unit for digital transformation efforts.

2. **Page 16-17 (Eastern and Southern Africa Section)**:
   - **Right Side**: A group of people standing on a paved street, some using mobile phones. It seems to be a casual, evening setting.
   - **Text**: Information about improving access to electricity in Rwanda and efforts for education and other services in Eastern and Southern Africa.

3. **Page 4-5 (Driving Action, Measuring Results)**:
   - **Images**: Various circular images and icons accompany text h

### 结论

在本 cookbook 中，我们着手增强面向富含图像、图形和表格的文档的检索增强生成（RAG）系统。传统 RAG 模型虽然擅长处理文本数据，却常常忽视通过视觉元素传达的丰富信息。通过集成视觉模型并利用元数据标记，我们弥合了这一差距，使 AI 能够有效地解读和利用视觉内容。

我们首先使用 Pinecone 设置了向量存储，为高效存储和检索向量嵌入奠定了基础。借助 GPT-4o 视觉模态解析 PDF 并提取视觉信息，我们将文档页面转换为相关文本。通过生成嵌入并标记包含视觉内容的页面，我们在向量存储中建立了一套健壮的元数据过滤体系。

将这些嵌入上传到 Pinecone，使其能够与我们的 RAG 处理工作流无缝集成。通过语义搜索，我们检索到与用户查询相匹配的相关页面，确保文本信息和视觉信息都得到考虑。通过将包含视觉内容的页面传给视觉模型来处理，我们提升了回答的准确性和深度，尤其是对于依赖图像或表格的查询。

我们以世界银行的 **A Better Bank for a Better World: Annual Report 2024** 为示例，演示了这些技术如何协同处理和解读复杂文档。这种方法不仅丰富了提供给用户的信息，还通过提供更全面、更准确的回答，显著提升了用户满意度和参与度。

遵循本 cookbook 中概述的概念，你现在已经能够构建可处理和解读包含复杂视觉元素的文档的 RAG 系统。这一进展为视觉数据扮演关键角色的各个领域的 AI 应用开辟了新的可能。